# Physics-Informed Neural Network (PINN) implemented in TensorFlow 2 to model 2D fluid flow around a cylinder.
 The model combines sparse velocity data with Navier-Stokes equations to esitmate flow parameters (λ₁, λ₂) and reconstruct the flow field.


**Data:** `main/Data/cylinder_nektar_wake.mat` (and vorticity reference `.mat` for figures).


In [ ]:
from __future__ import annotations

import logging
import os
import sys
import time
from itertools import combinations, product
from pathlib import Path

import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import scipy.io
import scipy.optimize
import tensorflow as tf
from mpl_toolkits.axes_grid1 import make_axes_locatable
from scipy.interpolate import griddata


def resolve_repo_paths():
    """Locate Utilities/, main/Data/, and this demo folder from cwd (walk parents)."""
    here = Path.cwd().resolve()
    for root in [here, *here.parents]:
        utils = root / "Utilities" / "plotting.py"
        data = root / "main" / "Data" / "2d_cylinder_wake.mat"
        ns_dir = root / "main" / "continuous_time_identification (Navier-Stokes)"
        if utils.is_file() and data.is_file() and ns_dir.is_dir():
            return ns_dir, root / "Utilities", root / "main" / "Data"
    raise FileNotFoundError(
        "Could not find Utilities/plotting.py and main/Data/cylinder_nektar_wake.mat. "
        "Run the notebook with cwd inside the NavierStokes-PINN repo."
    )


BASE_DIR, UTILS_DIR, DATA_DIR = resolve_repo_paths()
if str(UTILS_DIR) not in sys.path:
    sys.path.insert(0, str(UTILS_DIR))

from plotting import export_vector_figure, open_single_axis_figure

logging.getLogger("tensorflow").setLevel(logging.ERROR)

np.random.seed(1234)
tf.random.set_seed(1234)

## Device (GPU if available)

In [ ]:
def configure_device():
    """Prefer GPU when available, otherwise fall back to CPU."""
    gpus = tf.config.list_physical_devices("GPU")
    if gpus:
        for gpu in gpus:
            try:
                tf.config.experimental.set_memory_growth(gpu, True)
            except RuntimeError:
                pass
        return "/GPU:0", gpus
    return "/CPU:0", []


DEVICE, GPUS = configure_device()
print("Using device:", DEVICE)
if GPUS:
    print("Visible GPUs:", [gpu.name for gpu in GPUS])
else:
    print("No GPU detected by TensorFlow; running on CPU.")

## NavierStokesPINN model

In [ ]:
class NavierStokesPINN:
    """Learn λ₁, λ₂ and a velocity–pressure field from data + NS residuals (TF2 / tapes)."""

    def __init__(self, x, y, t, u, v, layers):
        X = np.concatenate([x, y, t], 1)

        self.lb = tf.constant(X.min(0), dtype=tf.float32)
        self.ub = tf.constant(X.max(0), dtype=tf.float32)

        self.X = X
        self.x = X[:, 0:1].astype(np.float32)
        self.y = X[:, 1:2].astype(np.float32)
        self.t = X[:, 2:3].astype(np.float32)
        self.u = u.astype(np.float32)
        self.v = v.astype(np.float32)

        self.layers = layers
        self.weights, self.biases = self._init_layers(layers)

        self.lambda_1 = tf.Variable(0.0, dtype=tf.float32)
        self.lambda_2 = tf.Variable(0.0, dtype=tf.float32)

        self.vars_train = self.weights + self.biases + [self.lambda_1, self.lambda_2]
        self.optimizer_Adam = tf.keras.optimizers.Adam()

        self._xc = tf.constant(self.x, dtype=tf.float32)
        self._yc = tf.constant(self.y, dtype=tf.float32)
        self._tc = tf.constant(self.t, dtype=tf.float32)
        self._uc = tf.constant(self.u, dtype=tf.float32)
        self._vc = tf.constant(self.v, dtype=tf.float32)

        self._adam_step_traced = tf.function(self._adam_train_step_eager, reduce_retracing=True)
        self._predict_traced = tf.function(self._predict_forward, reduce_retracing=True)

    def _init_layers(self, layers):
        weights = []
        biases = []
        for l in range(0, len(layers) - 1):
            W = self._glorot_truncated_weights(shape=(layers[l], layers[l + 1]))
            b = tf.Variable(tf.zeros([1, layers[l + 1]], dtype=tf.float32), dtype=tf.float32)
            weights.append(W)
            biases.append(b)
        return weights, biases

    def _glorot_truncated_weights(self, shape):
        in_dim, out_dim = shape[0], shape[1]
        std = np.sqrt(2 / (in_dim + out_dim)).astype(np.float32)
        return tf.Variable(
            tf.random.truncated_normal([in_dim, out_dim], stddev=std),
            dtype=tf.float32,
        )

    def _mlp_forward(self, X):
        depth = len(self.weights) + 1
        H = 2.0 * (X - self.lb) / (self.ub - self.lb) - 1.0
        for l in range(0, depth - 2):
            W = self.weights[l]
            b = self.biases[l]
            H = tf.tanh(tf.matmul(H, W) + b)
        W = self.weights[-1]
        b = self.biases[-1]
        return tf.matmul(H, W) + b

    def _flow_and_residuals(self, x, y, t):
        """Return (u, v, p) and NS residual vectors; nested tapes for spatial/time derivatives."""
        lambda_1 = self.lambda_1
        lambda_2 = self.lambda_2

        with tf.GradientTape(persistent=True) as tape3:
            tape3.watch([x, y, t])
            with tf.GradientTape(persistent=True) as tape2:
                tape2.watch([x, y, t])
                with tf.GradientTape(persistent=True) as tape1:
                    tape1.watch([x, y, t])
                    psi_and_p = self._mlp_forward(tf.concat([x, y, t], axis=1))
                    psi = psi_and_p[:, 0:1]
                    p = psi_and_p[:, 1:2]

                u = tape1.gradient(psi, y)
                v = -tape1.gradient(psi, x)
                p_x = tape1.gradient(p, x)
                p_y = tape1.gradient(p, y)

            u_t = tape2.gradient(u, t)
            u_x = tape2.gradient(u, x)
            u_y = tape2.gradient(u, y)
            v_t = tape2.gradient(v, t)
            v_x = tape2.gradient(v, x)
            v_y = tape2.gradient(v, y)

        u_xx = tape3.gradient(u_x, x)
        u_yy = tape3.gradient(u_y, y)
        v_xx = tape3.gradient(v_x, x)
        v_yy = tape3.gradient(v_y, y)

        del tape1
        del tape2
        del tape3

        f_u = u_t + lambda_1 * (u * u_x + v * u_y) + p_x - lambda_2 * (u_xx + u_yy)
        f_v = v_t + lambda_1 * (u * v_x + v * v_y) + p_y - lambda_2 * (v_xx + v_yy)

        return u, v, p, f_u, f_v

    def _collocation_tensors(self):
        return self._xc, self._yc, self._tc, self._uc, self._vc

    def _adam_train_step_eager(self):
        with tf.GradientTape() as tape:
            loss = self.loss_fn(self._xc, self._yc, self._tc, self._uc, self._vc)
            grads = tape.gradient(loss, self.vars_train)
        pairs = [(g, v) for g, v in zip(grads, self.vars_train) if g is not None]
        self.optimizer_Adam.apply_gradients(pairs)
        return loss

    def _predict_forward(self, x, y, t):
        u_p, v_p, p_p, _, _ = self._flow_and_residuals(x, y, t)
        return u_p, v_p, p_p

    def loss_fn(self, x, y, t, u_obs, v_obs):
        u_p, v_p, _p, f_u, f_v = self._flow_and_residuals(x, y, t)
        return (
            tf.reduce_sum(tf.square(u_obs - u_p))
            + tf.reduce_sum(tf.square(v_obs - v_p))
            + tf.reduce_sum(tf.square(f_u))
            + tf.reduce_sum(tf.square(f_v))
        )

    def callback(self, loss, lambda_1, lambda_2):
        l1 = float(lambda_1.numpy() if hasattr(lambda_1, "numpy") else lambda_1)
        l2 = float(lambda_2.numpy() if hasattr(lambda_2, "numpy") else lambda_2)
        print("Loss: %.3e, l1: %.3f, l2: %.5f" % (loss, l1, l2))

    def scipy_optimizer_minimize(self, maxiter=8000, maxfun=15000):
        x, y, t, u_obs, v_obs = self._collocation_tensors()
        var_shapes = [tuple(int(d) for d in v.shape) for v in self.vars_train]
        var_sizes = [int(np.prod(s)) for s in var_shapes]

        def pack():
            return np.concatenate([tf.reshape(v, [-1]).numpy() for v in self.vars_train])

        def unpack(flat):
            idx = 0
            for var, shp, sz in zip(self.vars_train, var_shapes, var_sizes):
                w = flat[idx : idx + sz].reshape(shp).astype(np.float32)
                var.assign(w)
                idx += sz

        def objfun(flat):
            unpack(flat.astype(np.float64))
            with tf.GradientTape() as tape:
                loss = self.loss_fn(x, y, t, u_obs, v_obs)
                grads = tape.gradient(loss, self.vars_train)
            g_parts = []
            for g, sz in zip(grads, var_sizes):
                if g is None:
                    g_parts.append(np.zeros(sz, dtype=np.float64))
                else:
                    g_parts.append(tf.reshape(g, [-1]).numpy().astype(np.float64))
            return float(loss.numpy()), np.concatenate(g_parts)

        x0 = pack().astype(np.float64)
        res = scipy.optimize.minimize(
            objfun,
            x0,
            method="L-BFGS-B",
            jac=True,
            options={
                "maxiter": maxiter,
                "maxfun": maxfun,
                "maxcor": 50,
                "maxls": 50,
                "ftol": float(np.finfo(float).eps),
            },
        )
        print(
            "L-BFGS-B: nit=%d, nfev=%d, success=%s"
            % (res.nit, getattr(res, "nfev", -1), res.success)
        )
        unpack(res.x.astype(np.float64))
        loss_val = float(self.loss_fn(x, y, t, u_obs, v_obs).numpy())
        self.callback(loss_val, self.lambda_1, self.lambda_2)

    def train(
        self,
        nIter,
        print_every=100,
        run_lbfgs=False,
        use_tf_function=True,
        lbfgs_maxiter=8000,
        lbfgs_maxfun=15000,
    ):
        start_time = time.time()
        step = self._adam_step_traced if use_tf_function else self._adam_train_step_eager
        for it in range(nIter):
            loss = step()

            if it % print_every == 0:
                elapsed = time.time() - start_time
                lv = float(loss.numpy())
                print(
                    "It: %d, Loss: %.3e, l1: %.3f, l2: %.5f, Time: %.2f"
                    % (
                        it,
                        lv,
                        float(self.lambda_1.numpy()),
                        float(self.lambda_2.numpy()),
                        elapsed,
                    )
                )
                start_time = time.time()

        if run_lbfgs:
            print(
                "\n--- Adam finished. L-BFGS-B next (many loss/grad evals; "
                "no per-step prints — can take several minutes) ---\n"
            )
            t_lbfgs = time.time()
            self.scipy_optimizer_minimize(maxiter=lbfgs_maxiter, maxfun=lbfgs_maxfun)
            print("--- L-BFGS-B wall time: %.1f s ---\n" % (time.time() - t_lbfgs,))

    def predict(self, x_star, y_star, t_star):
        x = tf.constant(x_star.astype(np.float32), dtype=tf.float32)
        y = tf.constant(y_star.astype(np.float32), dtype=tf.float32)
        t = tf.constant(t_star.astype(np.float32), dtype=tf.float32)
        u_p, v_p, p_p = self._predict_traced(x, y, t)
        return u_p.numpy(), v_p.numpy(), p_p.numpy()

## Small plotting utilities

In [ ]:
def _scalar_field_pcolor(X_star, values, figure_num):
    """Interpolate scattered samples to a grid and draw a pseudocolor map (quick diagnostic)."""
    lo = X_star.min(0)
    hi = X_star.max(0)
    n = 200
    gx = np.linspace(lo[0], hi[0], n)
    gy = np.linspace(lo[1], hi[1], n)
    Xg, Yg = np.meshgrid(gx, gy)
    Z = griddata(X_star, values.ravel(), (Xg, Yg), method="cubic")
    plt.figure(figure_num)
    plt.pcolormesh(Xg, Yg, Z, shading="auto", cmap="viridis")
    plt.colorbar()


def _equalize_3d_limits(ax):
    lims = np.array([getattr(ax, f"get_{axis}lim")() for axis in "xyz"])
    spans = lims[:, 1] - lims[:, 0]
    centers = np.mean(lims, axis=1)
    half = max(np.abs(spans)) / 8.0
    for c, axis in zip(centers, "xyz"):
        getattr(ax, f"set_{axis}lim")(c - half, c + half)

## Hyperparameters 

Set `TRAIN_MODEL = True` once to train and write `figures/NavierStokes_cache.npz`; otherwise the notebook loads that cache for fast plots.

In [ ]:
N_train = 5000
ADAM_ITERATIONS = 30000
PRINT_EVERY = 500
USE_TF_FUNCTION = True
RUN_LBFGS = False
LBFGS_MAXITER = 8000
LBFGS_MAXFUN = 15000
FIG_DIR = str(BASE_DIR / "figures")
SAVE_FIGURES = True
GRID_NN = 256
CMAP_PRESSURE = "turbo"
CMAP_VORT = "RdBu_r"
CMAP_SLICE = "cividis"
TRAIN_MODEL = False
CACHE_PATH = str(BASE_DIR / "figures" / "NavierStokes_cache.npz")

layers = [3, 20, 20, 20, 20, 20, 20, 20, 20, 2]

data = scipy.io.loadmat(str(DATA_DIR / "2d_cylinder_wake.mat"))

U_star = data["U_star"]
P_star = data["p_star"]
t_time = data["t"]
X_star = data["X_star"]

N = X_star.shape[0]
T = t_time.shape[0]

XX = np.tile(X_star[:, 0:1], (1, T))
YY = np.tile(X_star[:, 1:2], (1, T))
TT = np.tile(t_time, (1, N)).T

UU = U_star[:, 0, :]
VV = U_star[:, 1, :]
PP = P_star

x = XX.flatten()[:, None]
y = YY.flatten()[:, None]
t = TT.flatten()[:, None]

u = UU.flatten()[:, None]
v = VV.flatten()[:, None]
p = PP.flatten()[:, None]

snap = np.array([100])
x_star = X_star[:, 0:1]
y_star = X_star[:, 1:2]
t_star = TT[:, snap]
u_star = U_star[:, 0, snap]
v_star = U_star[:, 1, snap]
p_star = P_star[:, snap]

## Train PINN or load cached predictions

In [ ]:
if TRAIN_MODEL:
    with tf.device(DEVICE):
        idx = np.random.choice(N * T, N_train, replace=False)
        x_train = x[idx, :]
        y_train = y[idx, :]
        t_train = t[idx, :]
        u_train = u[idx, :]
        v_train = v[idx, :]

        model = NavierStokesPINN(x_train, y_train, t_train, u_train, v_train, layers)
        model.train(
            ADAM_ITERATIONS,
            print_every=PRINT_EVERY,
            run_lbfgs=RUN_LBFGS,
            use_tf_function=USE_TF_FUNCTION,
            lbfgs_maxiter=LBFGS_MAXITER,
            lbfgs_maxfun=LBFGS_MAXFUN,
        )

        u_pred, v_pred, p_pred = model.predict(x_star, y_star, t_star)
        lambda_1_value = float(model.lambda_1.numpy())
        lambda_2_value = float(model.lambda_2.numpy())

        error_u = np.linalg.norm(u_star - u_pred, 2) / np.linalg.norm(u_star, 2)
        error_v = np.linalg.norm(v_star - v_pred, 2) / np.linalg.norm(v_star, 2)
        error_p = np.linalg.norm(p_star - p_pred, 2) / np.linalg.norm(p_star, 2)
        error_lambda_1 = np.abs(lambda_1_value - 1.0) * 100
        error_lambda_2 = np.abs(lambda_2_value - 0.01) / 0.01 * 100

        print("Error u: %e" % (error_u))
        print("Error v: %e" % (error_v))
        print("Error p: %e" % (error_p))
        print("Error l1: %.5f%%" % (error_lambda_1))
        print("Error l2: %.5f%%" % (error_lambda_2))

        lb = X_star.min(0)
        ub = X_star.max(0)
        nn = GRID_NN
        x_plot = np.linspace(lb[0], ub[0], nn)
        y_plot = np.linspace(lb[1], ub[1], nn)
        X, Y = np.meshgrid(x_plot, y_plot)
        UU_star = griddata(X_star, u_pred.flatten(), (X, Y), method="cubic")
        VV_star = griddata(X_star, v_pred.flatten(), (X, Y), method="cubic")
        PP_star = griddata(X_star, p_pred.flatten(), (X, Y), method="cubic")
        P_exact = griddata(X_star, p_star.flatten(), (X, Y), method="cubic")

        noise = 0.01
        u_train_noisy = u_train + noise * np.std(u_train) * np.random.randn(*u_train.shape)
        v_train_noisy = v_train + noise * np.std(v_train) * np.random.randn(*v_train.shape)

        model = NavierStokesPINN(x_train, y_train, t_train, u_train_noisy, v_train_noisy, layers)
        model.train(
            ADAM_ITERATIONS,
            print_every=PRINT_EVERY,
            run_lbfgs=RUN_LBFGS,
            use_tf_function=USE_TF_FUNCTION,
            lbfgs_maxiter=LBFGS_MAXITER,
            lbfgs_maxfun=LBFGS_MAXFUN,
        )

        lambda_1_value_noisy = float(model.lambda_1.numpy())
        lambda_2_value_noisy = float(model.lambda_2.numpy())
        error_lambda_1_noisy = np.abs(lambda_1_value_noisy - 1.0) * 100
        error_lambda_2_noisy = np.abs(lambda_2_value_noisy - 0.01) / 0.01 * 100

        print("Error l1: %.5f%%" % (error_lambda_1_noisy))
        print("Error l2: %.5f%%" % (error_lambda_2_noisy))

        os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
        np.savez(
            CACHE_PATH,
            x_train=x_train,
            y_train=y_train,
            t_train=t_train,
            x_star=x_star,
            y_star=y_star,
            t_star=t_star,
            u_star=u_star,
            v_star=v_star,
            p_star=p_star,
            u_pred=u_pred,
            v_pred=v_pred,
            p_pred=p_pred,
            X=X,
            Y=Y,
            UU_star=UU_star,
            VV_star=VV_star,
            PP_star=PP_star,
            P_exact=P_exact,
            lambda_1_value=lambda_1_value,
            lambda_2_value=lambda_2_value,
            lambda_1_value_noisy=lambda_1_value_noisy,
            lambda_2_value_noisy=lambda_2_value_noisy,
            error_u=error_u,
            error_v=error_v,
            error_p=error_p,
            error_lambda_1=error_lambda_1,
            error_lambda_2=error_lambda_2,
            error_lambda_1_noisy=error_lambda_1_noisy,
            error_lambda_2_noisy=error_lambda_2_noisy,
        )
        print("Saved cache:", CACHE_PATH)

else:
    if not os.path.exists(CACHE_PATH):
        raise FileNotFoundError(
            "Cache not found. Set TRAIN_MODEL=True once to generate it:\n%s" % CACHE_PATH
        )
    cache = np.load(CACHE_PATH)
    x_train = cache["x_train"]
    y_train = cache["y_train"]
    t_train = cache["t_train"]
    x_star = cache["x_star"]
    y_star = cache["y_star"]
    t_star = cache["t_star"]
    u_star = cache["u_star"]
    v_star = cache["v_star"]
    p_star = cache["p_star"]
    u_pred = cache["u_pred"]
    v_pred = cache["v_pred"]
    p_pred = cache["p_pred"]
    X = cache["X"]
    Y = cache["Y"]
    UU_star = cache["UU_star"]
    VV_star = cache["VV_star"]
    PP_star = cache["PP_star"]
    P_exact = cache["P_exact"]
    lambda_1_value = float(cache["lambda_1_value"])
    lambda_2_value = float(cache["lambda_2_value"])
    lambda_1_value_noisy = float(cache["lambda_1_value_noisy"])
    lambda_2_value_noisy = float(cache["lambda_2_value_noisy"])
    error_u = float(cache["error_u"])
    error_v = float(cache["error_v"])
    error_p = float(cache["error_p"])
    error_lambda_1 = float(cache["error_lambda_1"])
    error_lambda_2 = float(cache["error_lambda_2"])
    error_lambda_1_noisy = float(cache["error_lambda_1_noisy"])
    error_lambda_2_noisy = float(cache["error_lambda_2_noisy"])

    print("Loaded cached predictions:", CACHE_PATH)
    print("Error u: %e" % (error_u))
    print("Error v: %e" % (error_v))
    print("Error p: %e" % (error_p))
    print("Error l1: %.5f%%" % (error_lambda_1))
    print("Error l2: %.5f%%" % (error_lambda_2))
    print("Error l1 noisy: %.5f%%" % (error_lambda_1_noisy))
    print("Error l2 noisy: %.5f%%" % (error_lambda_2_noisy))

## Figures: vorticity, collocation cloud, pressure

Saves PDF/EPS under `figures/` when `SAVE_FIGURES` is True.

In [ ]:
extent = [float(x_star.min()), float(x_star.max()), float(y_star.min()), float(y_star.max())]
U_exact = griddata(X_star, u_star.flatten(), (X, Y), method="cubic")
V_exact = griddata(X_star, v_star.flatten(), (X, Y), method="cubic")

dx = float(np.mean(np.diff(X[0, :])))
dy = float(np.mean(np.diff(Y[:, 0])))
w_pred = np.gradient(VV_star, dx, axis=1) - np.gradient(UU_star, dy, axis=0)
w_ref = np.gradient(V_exact, dx, axis=1) - np.gradient(U_exact, dy, axis=0)
w_err = np.abs(w_pred - w_ref)

w_valid = np.concatenate([w_pred[np.isfinite(w_pred)], w_ref[np.isfinite(w_ref)]])
w_abs = max(float(np.percentile(np.abs(w_valid), 98)), 1e-6) if w_valid.size else 1.0
err_valid = w_err[np.isfinite(w_err)]
w_err_hi = max(float(np.percentile(err_valid, 98)), 1e-6) if err_valid.size else 1.0
vort_norm = mcolors.TwoSlopeNorm(vmin=-w_abs, vcenter=0.0, vmax=w_abs)
box_lb = np.array([1.0, -2.0])
box_ub = np.array([8.0, 2.0])


def save_vorticity_figure(
    field,
    title,
    out_name,
    cmap,
    norm=None,
    vmin=None,
    vmax=None,
    cbar_label=r"$\omega$",
    draw_box=False,
):
    fig, ax = open_single_axis_figure(0.85, 1.0)
    h = ax.imshow(
        field,
        interpolation="bicubic",
        cmap=cmap,
        extent=extent,
        origin="lower",
        aspect="equal",
        norm=norm,
        vmin=vmin,
        vmax=vmax,
    )
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3.5%", pad=0.02)
    fig.colorbar(h, cax=cax, fraction=0.046, label=cbar_label)
    if draw_box:
        ax.plot([box_lb[0], box_lb[0]], [box_lb[1], box_ub[1]], "k", linewidth=1)
        ax.plot([box_ub[0], box_ub[0]], [box_lb[1], box_ub[1]], "k", linewidth=1)
        ax.plot([box_lb[0], box_ub[0]], [box_lb[1], box_lb[1]], "k", linewidth=1)
        ax.plot([box_lb[0], box_ub[0]], [box_ub[1], box_ub[1]], "k", linewidth=1)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_title(title, fontsize=11)
    if SAVE_FIGURES:
        os.makedirs(FIG_DIR, exist_ok=True)
        export_vector_figure(os.path.join(FIG_DIR, out_name), figure=fig)


data_vort = scipy.io.loadmat(str(DATA_DIR / "2d_cylinder_t0_vorticity.mat"))
x_vort = data_vort["x"]
y_vort = data_vort["y"]
w_vort = data_vort["w"]
modes = np.asarray(data_vort["modes"]).item()
nel = np.asarray(data_vort["nel"]).item()
xx_vort = np.reshape(x_vort, (modes + 1, modes + 1, nel), order="F")
yy_vort = np.reshape(y_vort, (modes + 1, modes + 1, nel), order="F")
ww_vort = np.reshape(w_vort, (modes + 1, modes + 1, nel), order="F")

w_abs_mat = max(float(np.percentile(np.abs(ww_vort.ravel()), 98)), 1e-6)
vort_norm_mat = mcolors.TwoSlopeNorm(vmin=-w_abs_mat, vcenter=0.0, vmax=w_abs_mat)
fig, ax = open_single_axis_figure(0.95, 1.0)
h = None
for i in range(0, nel):
    h = ax.pcolormesh(
        xx_vort[:, :, i],
        yy_vort[:, :, i],
        ww_vort[:, :, i],
        cmap=CMAP_VORT,
        norm=vort_norm_mat,
        shading="auto",
        rasterized=True,
    )
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="3.5%", pad=0.02)
fig.colorbar(h, cax=cax, fraction=0.046, label=r"$\omega$")
ax.plot([box_lb[0], box_lb[0]], [box_lb[1], box_ub[1]], "k", linewidth=1)
ax.plot([box_ub[0], box_ub[0]], [box_lb[1], box_ub[1]], "k", linewidth=1)
ax.plot([box_lb[0], box_ub[0]], [box_lb[1], box_lb[1]], "k", linewidth=1)
ax.plot([box_lb[0], box_ub[0]], [box_ub[1], box_ub[1]], "k", linewidth=1)
ax.set_aspect("equal", "box")
ax.set_xlabel("$x$")
ax.set_ylabel("$y$")
ax.set_title("Reference vorticity ($t=0$)", fontsize=11)
if SAVE_FIGURES:
    os.makedirs(FIG_DIR, exist_ok=True)
    export_vector_figure(os.path.join(FIG_DIR, "NavierStokes_vorticity"), figure=fig)
save_vorticity_figure(
    w_pred,
    r"Predicted vorticity $\omega$",
    "NavierStokes_vorticity_predicted",
    CMAP_VORT,
    norm=vort_norm,
)
save_vorticity_figure(
    w_ref,
    r"Reference vorticity $\omega$",
    "NavierStokes_vorticity_reference",
    CMAP_VORT,
    norm=vort_norm,
)
save_vorticity_figure(
    w_err,
    r"Absolute vorticity error",
    "NavierStokes_vorticity_absolute_error",
    "magma",
    vmin=0.0,
    vmax=w_err_hi,
    cbar_label=r"$|\omega_{\mathrm{pred}} - \omega_{\mathrm{ref}}|$",
)

fig, ax = open_single_axis_figure(1.0, 1.0)
ax.axis("off")
gs1 = gridspec.GridSpec(1, 2)
gs1.update(top=1.0, bottom=0.0, left=0.01, right=0.99, wspace=0)
ax = plt.subplot(gs1[:, 0], projection="3d")
ax.axis("off")

r1 = [x_star.min(), x_star.max()]
r2 = [t_time.min(), t_time.max()]
r3 = [y_star.min(), y_star.max()]

for s, e in combinations(np.array(list(product(r1, r2, r3))), 2):
    if (
        np.sum(np.abs(s - e)) == r1[1] - r1[0]
        or np.sum(np.abs(s - e)) == r2[1] - r2[0]
        or np.sum(np.abs(s - e)) == r3[1] - r3[0]
    ):
        ax.plot3D(*zip(s, e), color="k", linewidth=0.5)

ax.scatter(x_train, t_train, y_train, s=0.12, c=t_train.ravel(), cmap=CMAP_SLICE, alpha=0.85, linewidths=0)
ax.contourf(X, UU_star, Y, zdir="y", offset=float(np.mean(t_star)), cmap=CMAP_SLICE, alpha=0.9, levels=28)

ax.text(x_star.mean(), t_time.min() - 1, y_star.min() - 1, "$x$")
ax.text(x_star.max() + 1, t_time.mean(), y_star.min() - 1, "$t$")
ax.text(x_star.min() - 1, t_time.min() - 0.5, y_star.mean(), "$y$")
ax.text(x_star.min() - 3, t_time.mean(), y_star.max() + 1, "$u(t,x,y)$")
ax.set_xlim3d(r1)
ax.set_ylim3d(r2)
ax.set_zlim3d(r3)
_equalize_3d_limits(ax)

ax = plt.subplot(gs1[:, 1], projection="3d")
ax.axis("off")

r1 = [x_star.min(), x_star.max()]
r2 = [t_time.min(), t_time.max()]
r3 = [y_star.min(), y_star.max()]

for s, e in combinations(np.array(list(product(r1, r2, r3))), 2):
    if (
        np.sum(np.abs(s - e)) == r1[1] - r1[0]
        or np.sum(np.abs(s - e)) == r2[1] - r2[0]
        or np.sum(np.abs(s - e)) == r3[1] - r3[0]
    ):
        ax.plot3D(*zip(s, e), color="k", linewidth=0.5)

ax.scatter(x_train, t_train, y_train, s=0.12, c=t_train.ravel(), cmap=CMAP_SLICE, alpha=0.85, linewidths=0)
ax.contourf(X, VV_star, Y, zdir="y", offset=float(np.mean(t_star)), cmap=CMAP_SLICE, alpha=0.9, levels=28)

ax.text(x_star.mean(), t_time.min() - 1, y_star.min() - 1, "$x$")
ax.text(x_star.max() + 1, t_time.mean(), y_star.min() - 1, "$t$")
ax.text(x_star.min() - 1, t_time.min() - 0.5, y_star.mean(), "$y$")
ax.text(x_star.min() - 3, t_time.mean(), y_star.max() + 1, "$v(t,x,y)$")
ax.set_xlim3d(r1)
ax.set_ylim3d(r2)
ax.set_zlim3d(r3)
_equalize_3d_limits(ax)

if SAVE_FIGURES:
    os.makedirs(FIG_DIR, exist_ok=True)
    export_vector_figure(os.path.join(FIG_DIR, "NavierStokes_data"))

p_err = np.abs(PP_star - P_exact)
p_lo = float(np.percentile(np.minimum(PP_star, P_exact), 2))
p_hi = float(np.percentile(np.maximum(PP_star, P_exact), 98))
err_hi = float(np.percentile(p_err, 98))


def save_pressure_figure(field, title, out_name, cmap, vmin, vmax, cbar_label=None):
    fig, ax = open_single_axis_figure(0.8, 1.0)
    h = ax.imshow(
        field,
        interpolation="bicubic",
        cmap=cmap,
        extent=extent,
        origin="lower",
        aspect="equal",
        vmin=vmin,
        vmax=vmax,
    )
    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="3.5%", pad=0.02)
    if cbar_label is None:
        fig.colorbar(h, cax=cax, fraction=0.046)
    else:
        fig.colorbar(h, cax=cax, fraction=0.046, label=cbar_label)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$y$")
    ax.set_title(title, fontsize=11)
    if SAVE_FIGURES:
        export_vector_figure(os.path.join(FIG_DIR, out_name), figure=fig)


save_pressure_figure(
    PP_star,
    r"Predicted $p$",
    "NavierStokes_predicted_p",
    CMAP_PRESSURE,
    p_lo,
    p_hi,
)
save_pressure_figure(
    P_exact,
    r"Reference $p$",
    "NavierStokes_reference_p",
    CMAP_PRESSURE,
    p_lo,
    p_hi,
)
save_pressure_figure(
    p_err,
    r"Absolute error",
    "NavierStokes_absolute_error",
    "magma",
    0.0,
    err_hi,
    cbar_label=r"$|p_{\mathrm{pred}} - p_{\mathrm{ref}}|$",
)